In [0]:
import json
from datetime import datetime

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_metadata", "")

run_id = dbutils.widgets.get("run_id")
table_metadata = dbutils.widgets.get("table_metadata")

if not run_id or not table_metadata:
    raise ValueError("run_id and table_metadata are required")

table_metadata = json.loads(table_metadata.replace("'", '"'))

table_id = table_metadata["table_id"]
table_name = table_metadata["table_name"]

start_time = datetime.utcnow()

print("Run ID:", run_id)
print("Table ID:", table_id)
print("Table Name:", table_name)

Run ID: 12345
Table ID: 1
Table Name: customers


/home/spark-49b8b9d1-70a6-4f7d-ab14-e3/.ipykernel/63/command-6441222018925173-3715939592:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()


In [0]:
%sql
create schema if not exists banking.gold;

In [0]:
entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""").count() > 0

if entry_exists:
    
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            layer = 'Gold',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """)

else:

    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Gold',
            TIMESTAMP('{start_time}'),
            NULL,
            'INPROGRESS',
            NULL,
            NULL
        )
    """)

print("Audit entry created / updated")


Audit entry created / updated


In [0]:
notebook_path = f"gold_transformations/{table_name}"

print("Notebook to execute:", notebook_path)

Notebook to execute: gold_transformations/customers


In [0]:
status = "SUCCESS"
error_message = None
records = None

try:

    result = dbutils.notebook.run(
        notebook_path,
        timeout_seconds=0
    )

    # Expect the notebook to return record count
    if result:
        records = int(result)

    print("Notebook completed successfully")
    print("Records:", records)

except Exception as e:

    status = "FAILED"
    error_message = str(e)

    print("Notebook failed")
    print(error_message)


Notebook failed
An error occurred while calling NotebookRun.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Unable to access the notebook "/Users/ranayuvrajsingh1111@gmail.com/banking_etl_project/03_Silver_to_Gold/gold_transformations/customers". Either it does not exist, or the identity used to run this job, ranayuvrajsingh1111@gmail.com, lacks the required permissions.
	at com.databricks.workflow.WorkflowDriver.run(WorkflowDriver.scala:92)
	at com.databricks.dbutils_v1.impl.NotebookUtilsImpl.run(NotebookUtilsImpl.scala:137)
	at com.databricks.backend.daemon.driver.PythonDriverLocalBase$PythonDriverLocalGrpcInterfaceImpl.notebookRun(PythonDriverLocalBase.scala:1404)
	at com.databricks.backend.daemon.repl.ExecutionLifecycleService.$anonfun$notebookRun$3(ExecutionLifecycleService.scala:201)
	at com.databricks.backend.daemon.repl.ExecutionLifecycleService.translateUserErrors(ExecutionLifecycleService.scala:59)
	at com.databricks.backend.daemon.repl

In [0]:
end_time = datetime.utcnow()

spark.sql(f"""
    UPDATE banking.metadata.pipeline_runs
    SET
        end_time = TIMESTAMP('{end_time}'),
        status = '{status}',
        number_of_records = {records if records else 'NULL'},
        error_message = {f"'{error_message}'" if error_message else 'NULL'}
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""")

print("Audit table updated")

In [0]:
if status == "FAILED":
    raise Exception(error_message)